In [1]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
import sys
import os


In [2]:
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for Jupyter Notebooks
    notebook_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))
from pcp_project.pipeline import SubjectPipeline

In [3]:
# Mock estimators

class BandPassFilter(BaseEstimator, TransformerMixin):
    def __init__(self, frequency_bands):
        self.frequency_bands = frequency_bands
        
    def fit(self, X, y=None, **kwargs):
        return self
        
    def transform(self, X, y=None):
        return X

class StateSelector(BaseEstimator, TransformerMixin):
    def __init__(self, states):
        self.states = states
        
    def fit(self, X, y=None, **kwargs):
        return self
        
    def fit_transform(self, X, y=None, groups=None):
        # The SubjectPipeline checks for tuple return to update its mask.
        return X, groups
        
    def transform(self, X, y=None, groups=None):
        return X

class SlidingWindow(BaseEstimator, TransformerMixin):
    def __init__(self, length, step_size):
        self.length = length
        self.step_size = step_size
        
    def fit(self, X, y=None, **kwargs):
        return self
        
    def transform(self, X, y=None):
        # Mock epoching: Convert (n_samples, channels) to (n_epochs, channels, times)
        # We will add an epoch dimension and transpose to (channels, times)
        if isinstance(X, list) or (isinstance(X, np.ndarray) and X.dtype == object):
            # If X is a list of arrays (multiple subjects), epoch each subject
            return np.array([np.expand_dims(subj_x.T, axis=0) for subj_x in X])
        return np.expand_dims(X.T, axis=0)

class BatchedOAS(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None, **kwargs):
        return self
        
    def transform(self, X, y=None):
        # Mock covariance
        if isinstance(X, np.ndarray) and X.ndim == 4:
            # Batch of subjects
            n_subj, n_epochs, channels, _ = X.shape
            return np.zeros((n_subj, n_epochs, channels, channels))
        n_epochs, channels, _ = X.shape
        return np.zeros((n_epochs, channels, channels))

try:
    from pyriemann.tangentspace import TangentSpace
except ImportError:
    class TangentSpace(BaseEstimator, TransformerMixin):
        def fit(self, X, y=None, **kwargs):
            return self
        def transform(self, X, y=None):
            # Mock projection: (..., channels, channels) -> (..., features)
            if X.ndim == 4:
                n_subj, n_epochs, channels, _ = X.shape
                features = channels * (channels + 1) // 2
                return np.zeros((n_subj, n_epochs, features))
            n_epochs, channels, _ = X.shape
            features = channels * (channels + 1) // 2
            return np.zeros((n_epochs, features))

class FeatureConsolidator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None, **kwargs):
        return self
        
    def transform(self, X, y=None):
        # Consolidate epoch features into a single subject-level feature vector
        if X.ndim == 3:
            return np.mean(X, axis=1) # Mean across epochs for each subject
        return np.mean(X, axis=0, keepdims=True)

class MeanProbabilityAggregator(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None):
        self.estimator = estimator or LogisticRegression()
        
    def fit(self, X, y):
        # Flatten subjects and epochs for the classifier
        n_subj, n_epochs, n_features = X.shape
        X_flat = X.reshape(-1, n_features)
        y_flat = np.repeat(y, n_epochs)
        self.estimator.fit(X_flat, y_flat)
        return self
        
    def predict(self, X):
        n_subj, n_epochs, n_features = X.shape
        X_flat = X.reshape(-1, n_features)
        preds = self.estimator.predict(X_flat)
        # Aggregate back to subject level (mocking mean over epochs)
        preds_reshaped = preds.reshape(n_subj, n_epochs)
        # Just taking the mode or first prediction for infrastructure mock
        return preds_reshaped[:, 0]

In [4]:
import csv
labels_path = os.path.join(notebook_dir, '..', 'data', 'labels_reduced.csv')

# Read the CSV and find the label for AD_001
# We'll use SCID5_CV_Depression as the target disorder
target_disorder = "SCID5_CV_Depression"
ad_001_label = 0.0
with open(labels_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['EEG_ID'] == 'AD_001':
            ad_001_label = float(row[target_disorder])
            break
    

In [5]:
data_path = os.path.join(notebook_dir, '..', 'data', 'AD_001.npz')
data = np.load(data_path)

X_subject = data['X']
# (1: eyes_closed, 0: eyes_open)
y_state = data['y'][0] if data['y'].ndim > 1 else data['y']

# instead of loading all subjects, 
# dummy dataset with 4 subjects to make cross_validation work (needs >=2 classes per fold)
dataset = np.array([X_subject, X_subject, X_subject, X_subject], dtype=object)

# mock labels for the dummy subjects 
labels = np.array([ad_001_label, 1.0 - ad_001_label, ad_001_label, 1.0 - ad_001_label])


In [6]:
mask_eyes_closed = (y_state == 1)
subject_masks = np.array([mask_eyes_closed, mask_eyes_closed, mask_eyes_closed, mask_eyes_closed], dtype=object)

In [7]:
# option 1 with feature consolidator
pipe = SubjectPipeline(
        steps=[
            ("filter", BandPassFilter(frequency_bands=[[5, 10], [13, 35]])),
            ("select", StateSelector(states=["eyes_closed"])),
            ("epoch", SlidingWindow(length=200, step_size=50)),
            ("covariance", BatchedOAS()),
            ("features", TangentSpace()), 
            ("consolidator", FeatureConsolidator()), # option 1
            ("clf", LogisticRegression()), 
        ],
        mask=subject_masks # argument 
    )

In [8]:
print("Target Disorder:", target_disorder)
print("AD_001 Label:", ad_001_label)
    
scores1 = cross_validate(pipe, X=dataset, y=labels, cv=2)
print("Option 1 Scores:", scores1)

Target Disorder: SCID5_CV_Depression
AD_001 Label: 0.0
Option 1 Scores: {'fit_time': array([1.65294981, 1.35470796]), 'score_time': array([1.30862117, 1.15996504]), 'test_score': array([0.5, 0.5])}
